# LoveDA: exploratory data analysis

Everything here is about one question: **what makes this dataset hard, and what does that
imply for point supervision?**

The short version is class imbalance. LoveDA is imbalanced enough that a uniformly sampling
annotator will simply never label some classes, and that single fact explains most of the
results in the README.

> Needs the dataset. See the README: put `Train.zip` and `Val.zip` in `data/`.
> Nothing is unzipped; images are read straight out of the archives.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

from pointseg.config import CLASSES, NC, IGNORE, PALETTE
from pointseg.data import list_split, read_image, read_mask, colorize
from pointseg.points import sample_points, label_distribution

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': False, 'font.size': 11})
LEGEND = [mpatches.Patch(facecolor=PALETTE[i]/255, edgecolor='k', label=c)
          for i, c in enumerate(CLASSES)]

train_items, test_items = list_split('Train'), list_split('Val')
print(f'train {len(train_items)} images, test {len(test_items)} images')


## 1. How much data, and from where

LoveDA splits into **Rural** and **Urban** domains. They look different enough that the
domain gap is part of what the benchmark is testing, so it is worth knowing the balance
before reading any per-class number.


In [ ]:
from collections import Counter
cnt_tr = Counter(i['domain'] for i in train_items)
cnt_te = Counter(i['domain'] for i in test_items)

fig, ax = plt.subplots(figsize=(6, 3.6))
x = np.arange(2); w = 0.38
ax.bar(x - w/2, [cnt_tr['Rural'], cnt_tr['Urban']], w, label='Train')
ax.bar(x + w/2, [cnt_te['Rural'], cnt_te['Urban']], w, label='Val')
ax.set_xticks(x); ax.set_xticklabels(['Rural', 'Urban'])
ax.set_ylabel('images'); ax.legend(); ax.grid(axis='y', alpha=.25)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
print(dict(cnt_tr), dict(cnt_te))


## 2. What the images and masks look like

0.3 m aerial imagery at 1024x1024. Note the black regions in the overlay: LoveDA marks
**no-data** with label 0, which `read_mask` maps to `IGNORE`. Those pixels are excluded from
the loss and from every metric, and they are never sampled as points.


In [ ]:
rng = np.random.default_rng(1)
picks = rng.choice(len(train_items), 3, replace=False)
fig, ax = plt.subplots(3, 3, figsize=(13, 13))
for r, i in enumerate(picks):
    it = train_items[i]
    im, m = read_image(it['image']), read_mask(it['mask'])
    ax[r,0].imshow(im);            ax[r,0].set_title(f"{it['id']}  image")
    ax[r,1].imshow(colorize(m));   ax[r,1].set_title('mask')
    ax[r,2].imshow(im); ax[r,2].imshow(colorize(m), alpha=.55); ax[r,2].set_title('overlay')
for a in ax.ravel(): a.axis('off')
fig.legend(handles=LEGEND, loc='lower center', ncol=NC, frameon=False)
plt.tight_layout(rect=[0,0.04,1,1]); plt.show()


## 3. Class imbalance, the central problem

Computed once over the training masks and reused below. This is the cell that explains the
whole project, so it is worth the wait.


In [ ]:
stats = []
for it in tqdm(train_items, desc='class stats'):
    m = read_mask(it['mask']).ravel()
    valid = m[m != IGNORE]
    stats.append(dict(domain=it['domain'],
                      counts=np.bincount(valid, minlength=NC),
                      nodata=float((m == IGNORE).mean())))

counts = np.stack([s['counts'] for s in stats])
total = counts.sum(0)
share = total / total.sum()
present = (counts > 0).mean(0)          # fraction of images containing the class
print(f'mean no-data fraction: {np.mean([s["nodata"] for s in stats]):.2%}')


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 4.4))

ax[0].bar(CLASSES, share, color=PALETTE/255, edgecolor='k', lw=.6)
ax[0].set_yscale('log'); ax[0].set_ylabel('share of all labelled pixels (log)')
ax[0].set_title('Pixel share per class')
for i, v in enumerate(share):
    ax[0].text(i, v*1.15, f'{v:.1%}', ha='center', fontsize=8)

ax[1].bar(CLASSES, present, color=PALETTE/255, edgecolor='k', lw=.6)
ax[1].set_ylabel('fraction of images containing it'); ax[1].set_ylim(0,1)
ax[1].set_title('How often each class appears at all')

# within the images that DO contain it, how much of the image is it?
with np.errstate(invalid='ignore'):
    frac = counts / np.maximum(counts.sum(1, keepdims=True), 1)
within = np.array([frac[counts[:,c] > 0, c].mean() if (counts[:,c] > 0).any() else 0
                   for c in range(NC)])
ax[2].bar(CLASSES, within, color=PALETTE/255, edgecolor='k', lw=.6)
ax[2].set_ylabel('mean area share where present')
ax[2].set_title('Rare classes are rare AND small')

for a in ax:
    a.tick_params(axis='x', rotation=35)
    a.grid(axis='y', alpha=.25); a.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

for c, s, p, w in zip(CLASSES, share, present, within):
    print(f'{c:12} {s:7.2%} of pixels | in {p:5.1%} of images | {w:6.2%} of those images')


### Why this decides everything downstream

Two numbers to hold onto:

- **background dominates.** It is over half of all labelled pixels on its own, which is why
  overall accuracy is a misleading metric here and mIoU is the one the README reports.
- **`barren` is both rare and small.** It appears in a minority of images and occupies very
  little of the ones it does appear in.

Now do the arithmetic for a uniform annotator. If a class is 0.5% of an image's pixels and
you click 20 random points, the chance of hitting it even once is about
$1 - 0.995^{20} \approx 10\%$. **Nine times out of ten that class receives no supervision
at all in that image.** That is the failure class-balanced sampling exists to fix.


## 4. What each class actually looks like


In [ ]:
samples = {c: [] for c in range(NC)}
for it in train_items[::40]:
    m = read_mask(it['mask'])
    for c in range(NC):
        if len(samples[c]) < 1 and (m == c).mean() > 0.25:
            samples[c].append(it)

fig, ax = plt.subplots(2, NC, figsize=(22, 6.5))
for c in range(NC):
    ax[0,c].set_title(CLASSES[c], color=PALETTE[c]/255 if c else 'k', fontweight='bold')
    if samples[c]:
        it = samples[c][0]
        ax[0,c].imshow(read_image(it['image']))
        ax[1,c].imshow(colorize(read_mask(it['mask'])))
    else:
        ax[0,c].text(.5,.5,'no image is\n>25% this class', ha='center', va='center')
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()


## 5. What a point label actually is

This is the heart of it. The mask below has **1,048,576** labelled pixels. The point targets
have twenty. Points are dilated purely so they are visible on screen; the real targets are
single pixels.


In [ ]:
import cv2
it = train_items[7]
im, m = read_image(it['image']), read_mask(it['mask'])

fig, ax = plt.subplots(1, 3, figsize=(18, 6.2))
ax[0].imshow(im); ax[0].imshow(colorize(m), alpha=.55)
ax[0].set_title(f'dense mask: {(m != IGNORE).sum():,} labelled pixels')

for k, strategy in enumerate(['uniform', 'class_balanced'], start=1):
    ys, xs, labels = sample_points(m, 20, strategy, np.random.default_rng(0))
    canvas = np.full(m.shape, IGNORE, np.uint8); canvas[ys, xs] = labels
    big = cv2.dilate(np.where(canvas == IGNORE, 0, canvas + 1).astype(np.uint8),
                     np.ones((15, 15), np.uint8))
    ax[k].imshow(im * 0.35 / 255)
    ov = colorize(np.where(big == 0, IGNORE, big - 1).astype(np.uint8))
    ax[k].imshow(ov, alpha=(big > 0).astype(float))
    ax[k].set_title(f'{strategy}: {len(labels)} labelled pixels\n'
                    f'classes covered: {sorted(set(labels.tolist()))}')
for a in ax: a.axis('off')
fig.legend(handles=LEGEND, loc='lower center', ncol=NC, frameon=False)
plt.tight_layout(rect=[0,0.05,1,1]); plt.show()


## 6. Which classes does each strategy actually label?

Same annotation budget of 20 points per image, across the whole training split. The only
difference is how the budget is allocated. This chart is the most direct evidence for the
main result in the README.


In [ ]:
subset = train_items[::10]      # a tenth of the split is plenty for the distribution
dist = {}
for strategy in ('uniform', 'class_balanced'):
    rng = np.random.default_rng(0)
    pts = {it['id']: sample_points(read_mask(it['mask']), 20, strategy, rng)
           for it in tqdm(subset, desc=strategy, leave=False)}
    dist[strategy] = label_distribution(pts, NC)

fig, ax = plt.subplots(1, 2, figsize=(16, 4.4))
x = np.arange(NC); w = 0.38
ax[0].bar(x - w/2, dist['uniform'], w, label='uniform')
ax[0].bar(x + w/2, dist['class_balanced'], w, label='class-balanced')
ax[0].set_yscale('log'); ax[0].set_ylabel('labelled points (log)')
ax[0].set_xticks(x); ax[0].set_xticklabels(CLASSES, rotation=35, ha='right')
ax[0].set_title('Where the same 20-point budget goes'); ax[0].legend()

ratio = dist['class_balanced'] / np.maximum(dist['uniform'], 1)
ax[1].bar(x, ratio, color=PALETTE/255, edgecolor='k', lw=.6)
ax[1].axhline(1, c='k', lw=.8, ls='--')
ax[1].set_yscale('log'); ax[1].set_ylabel('balanced / uniform')
ax[1].set_xticks(x); ax[1].set_xticklabels(CLASSES, rotation=35, ha='right')
ax[1].set_title('Fold increase in supervision per class')
for a in ax:
    a.grid(axis='y', alpha=.25); a.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

print(f"{'class':12}{'uniform':>12}{'balanced':>12}{'ratio':>9}")
for c, u, b, r in zip(CLASSES, dist['uniform'], dist['class_balanced'], ratio):
    print(f'{c:12}{u:12,}{b:12,}{r:9.1f}x')


## 7. Does the budget even get spent?

A detail worth checking rather than assuming. `class_balanced` guarantees every present
class at least one point, but a class with fewer pixels than its allocated share contributes
everything it has and the total comes in **under** the nominal budget. Sampling with
replacement instead would duplicate supervision and silently reweight the loss.


In [ ]:
for strategy in ('uniform', 'class_balanced'):
    rng = np.random.default_rng(0)
    totals = [len(sample_points(read_mask(it['mask']), 20, strategy, rng)[2])
              for it in tqdm(train_items[::25], desc=strategy, leave=False)]
    totals = np.array(totals)
    print(f'{strategy:16} mean {totals.mean():5.2f} / 20   '
          f'min {totals.min()}  max {totals.max()}  '
          f'under budget: {(totals < 20).mean():.1%} of images')


## Takeaways

1. **LoveDA is severely imbalanced**, and background alone dominates the pixel count. That is
   why mIoU, not overall accuracy, is the metric the results are judged on.
2. **Rare classes are rare and small**, so a uniform annotator misses them in most images.
   `barren` is the clearest case.
3. **Class-balanced sampling moves supervision to exactly those classes** at identical
   annotation cost, which is the mechanism behind the main result: 20 balanced points beat
   500 uniform ones, and beat the full dense mask when combined with focal loss.

Numbers and figures for the trained runs: `pointseg-report`, or the README.
